# Predicting Student Health Risk

## Project Overview
Predict health status (fit / at-risk / unhealthy) from student health indicators.

## Feature Engineering Strategy (EDA-based)
1. sleep_duration: strongest predictor - fit(7.95h) > at-risk(7.09h) > unhealthy(5.37h)
2. stress_level: near-deterministic - high=97.5% unhealthy, low=20% fit, medium=99.4% at-risk
3. physical_activity: active=96.8% fit, sedentary=91.7% at-risk
4. bmi: unhealthy(24.1) > at-risk(23.0) > fit(21.8)
5. smoking_alcohol: yes=45% unhealthy, no=45% fit
6. sleep_quality: poor=54% unhealthy, good=46% fit
7. Numerical features nearly independent (corr ~ 0)
8. Missing rates consistent across classes (MCAR)

## Final Feature Set (46 features)
| Category | Count | Description |
|----------|-------|-------------|
| Raw Numeric | 7 | Original numeric columns |
| Binned OHE | 10 | BMI x5 + Heart Rate x5 |
| Simple Derived | 5 | Arithmetic combinations |
| Ordinal Encoded | 6 | Categories to 0/1/2 |
| Category OHE | 18 | 6 raw cats x 3 values each |

## Model
CatBoost (depth=6, lr=auto) + 5-Fold CV + Balanced Weights + Early Stopping


In [ ]:
# ============================================================
# 0. Imports & Configuration
# ============================================================
import pandas as pd
import numpy as np
import logging
import sys
import warnings
from datetime import datetime
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
# DATA_DIR = Path('input')  # local path


In [ ]:
# ============================================================
# Kaggle Environment Configuration
# ============================================================
import os
from pathlib import Path

# Paths - adjust for your environment
DATA_DIR = Path('/kaggle/input/competitions/playground-series-s6e7')
OUTPUT_DIR = Path('/kaggle/working/')

# Create output directory if needed
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Data path: {DATA_DIR}")
print(f"Output path: {OUTPUT_DIR}")


---
## Configuration
Set paths for Kaggle environment.
- Input: `/kaggle/input/competitions/playground-series-s6e7`
- Output: `/kaggle/working/`


## Step 1: Load Data


In [ ]:
logger.info("=" * 60)
logger.info("STEP 1: LOADING DATA")
logger.info("=" * 60)

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')

logger.info(f"Train shape: {train.shape}")
logger.info(f"Test shape: {test.shape}")
logger.info(f"Train columns: {list(train.columns)}")
logger.info(f"Test columns: {list(test.columns)}")


## Step 2: Exploratory Data Analysis

Key findings:
- Severe class imbalance: at-risk(85.9%), unhealthy(8.4%), fit(5.8%)
- Missing rates are MCAR (consistent across classes)
- Numerical features have near-zero correlation
- sleep_duration and stress_level are the strongest predictors


In [ ]:
logger.info("=" * 60)
logger.info("STEP 2: EXPLORATORY DATA ANALYSIS")
logger.info("=" * 60)

# 2a. Target distribution - confirm imbalance
logger.info(f"\nTarget distribution:\n{train['health_condition'].value_counts()}")
logger.info(f"\nTarget distribution (%):\n{train['health_condition'].value_counts(normalize=True).mul(100).round(2)}")


In [ ]:
# 2b. Missing value analysis
# MCAR - missing rates are similar across all 3 classes
logger.info("\n--- Missing Values ---")
for name, df in [('Train', train), ('Test', test)]:
    miss = df.isnull().sum()
    miss_df = pd.DataFrame({'count': miss, 'pct': df.isnull().mean().mul(100).round(2)})
    miss_df = miss_df[miss_df['count'] > 0].sort_values('count', ascending=False)
    logger.info(f"{name} missing:\n{miss_df}")


In [ ]:
# 2c. Define column groups and convert types
num_cols = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
            'step_count', 'exercise_duration', 'water_intake']
cat_cols = ['diet_type', 'stress_level', 'sleep_quality',
            'physical_activity_level', 'smoking_alcohol', 'gender']

# All CSV columns are object type - convert numeric columns to float
for col in num_cols:
    train[col] = pd.to_numeric(train[col], errors='coerce')
    test[col] = pd.to_numeric(test[col], errors='coerce')

logger.info(f"\nNumerical stats:\n{train[num_cols].describe().round(2)}")


In [ ]:
# 2d. Target vs numerical features (mean)
# Key insight: sleep_duration has largest separation
target_mean = train.groupby('health_condition')[num_cols].mean().round(2)
logger.info(f"\nTarget vs Numerical (mean):\n{target_mean}")


In [ ]:
# 2e. Categorical feature distributions
logger.info("\n--- Categorical Feature Distributions ---")
for col in cat_cols:
    vc = train[col].value_counts(dropna=False)
    logger.info(f"\n{col}:\n{vc}")


In [ ]:
# 2f. Target vs categorical features (row-wise %)
# Key: stress_level is near-deterministic for health status
logger.info("\n--- Target vs Categorical (% by row) ---")
for col in cat_cols:
    ct = pd.crosstab(train[col], train['health_condition'], normalize='index').mul(100).round(1)
    logger.info(f"\n{col}:\n{ct}")


## Step 3: Feature Engineering

Building 46 features based on EDA findings:
1. **Binned features**: Discretize continuous values at medical thresholds
2. **Simple derived**: Arithmetic combinations with domain meaning
3. **Ordinal encoding**: Preserve order information in categories
4. **One-Hot encoding**: Expand unordered categories


In [ ]:
# ============================================================
# 3. FEATURE ENGINEERING
# ============================================================
logger.info("=" * 60)
logger.info("STEP 3: FEATURE ENGINEERING")
logger.info("=" * 60)

def engineer_features(df, is_train=True):
    df = df.copy()
    
    # 3a. Binned features - discretize at medical thresholds
    # BMI categories (WHO standard)
    # EDA: obese + any activity = ~83% unhealthy
    #       normal + active = 17.6% fit
    # Binning captures threshold effects (BMI>25 overweight, BMI>30 obese)
    df['bmi_category'] = pd.cut(df['bmi'],
                                 bins=[0, 18.5, 25, 30, 100],
                                 labels=['underweight', 'normal', 'overweight', 'obese'])
    logger.info("[3a] bmi_category: discretized bmi -> underweight/normal/overweight/obese")
    
    # Heart rate zones (clinical standard)
    # EDA: mean HR is similar across classes (74.8-75.3)
    # But extreme values differ - binning captures risk signals
    df['heart_rate_zone'] = pd.cut(df['heart_rate'],
                                    bins=[0, 60, 80, 100, 300],
                                    labels=['low', 'normal', 'elevated', 'high'])
    logger.info("[3a] heart_rate_zone: discretized heart_rate -> low/normal/elevated/high")
    
    # 3b. Simple derived features (arithmetic combinations)
    # sleep_debt = 8 - sleep_duration (subtraction)
    # EDA: fit median 7.95h (~0 debt), unhealthy median 5.37h (debt=2.63)
    # Directly measures deviation from ideal sleep
    df['sleep_debt'] = 8 - df['sleep_duration']
    logger.info("[3b] sleep_debt: 8 - sleep_duration (subtraction)")
    
    # bmi_x_heartrate = bmi x heart_rate / 100 (multiplication)
    # EDA: unhealthy has higher BMI AND higher HR, product amplifies signal
    df['bmi_x_heartrate'] = df['bmi'] * df['heart_rate'] / 100
    logger.info("[3b] bmi_x_heartrate: bmi x heart_rate / 100 (multiplication)")
    
    # calorie_per_step = calorie / (step_count + 1) (division ratio)
    # EDA: fit steps(12342) >> at-risk(8483), unhealthy(8941)
    # Energy per step reflects metabolic efficiency independent of volume
    df['calorie_per_step'] = df['calorie_expenditure'] / (df['step_count'] + 1)
    logger.info("[3b] calorie_per_step: calorie / step_count (division ratio)")
    
    # exercise_intensity = calorie / (exercise_duration + 0.1) (division ratio)
    # EDA: fit exercise(50min) > at-risk(38min), unhealthy(39min)
    # Intensity per minute is more stable than absolute duration
    df['exercise_intensity'] = df['calorie_expenditure'] / (df['exercise_duration'] + 0.1)
    logger.info("[3b] exercise_intensity: calorie / exercise_duration (division ratio)")
    
    # water_per_kg = water / (bmi + 0.1) (division ratio)
    # EDA: absolute water intake is nearly identical across classes (~2.18L)
    # Relative to BMI reveals true hydration status
    df['water_per_kg'] = df['water_intake'] / (df['bmi'] + 0.1)
    logger.info("[3b] water_per_kg: water_intake / bmi (division ratio)")
    
    # 3c. Ordinal encoding - preserve order relationships
    # OHE loses low<medium<high ordering
    # Ordinal encoding lets CatBoost split on thresholds (e.g., stress_level_ord >= 1)
    # EDA: stress_level is near-deterministic, ordering is critical
    ord_maps = {
        'stress_level_ord': ('stress_level', {'low': 0, 'medium': 1, 'high': 2}),
        'sleep_quality_ord': ('sleep_quality', {'good': 2, 'average': 1, 'poor': 0}),
        'physical_activity_level_ord': ('physical_activity_level', {'sedentary': 0, 'moderate': 1, 'active': 2}),
        'smoking_alcohol_ord': ('smoking_alcohol', {'no': 0, 'occasional': 1, 'yes': 2}),
        'diet_type_ord': ('diet_type', {'veg': 0, 'balanced': 1, 'non-veg': 2}),
        'gender_ord': ('gender', {'female': 0, 'male': 1, 'other': 2}),
    }
    for new_col, (src, m) in ord_maps.items():
        df[new_col] = df[src].map(m)
    logger.info("[3c] Ordinal encoding: 6 categories mapped to 0/1/2")
    
    # 3d. Missing value imputation
    # MCAR - median for numeric, mode for categorical
    # Store fill values during training, reuse for test (no data leakage)
    for col in num_cols:
        if is_train:
            fill_val = df[col].median()
            setattr(engineer_features, f'{col}_fill', fill_val)
        else:
            fill_val = getattr(engineer_features, f'{col}_fill', df[col].median())
        df[col] = df[col].fillna(fill_val)
    
    for col in cat_cols:
        if is_train:
            fill_val = df[col].mode().iloc[0] if not df[col].mode().empty else 'unknown'
            setattr(engineer_features, f'{col}_fill', fill_val)
        else:
            fill_val = getattr(engineer_features, f'{col}_fill', 'unknown')
        df[col] = df[col].fillna(fill_val)
    
    # NaN from binning (original value was missing) -> 'unknown' category
    for col in ['bmi_category', 'heart_rate_zone']:
        if hasattr(df[col], 'cat'):
            df[col] = df[col].cat.add_categories('unknown')
        df[col] = df[col].fillna('unknown')
    
    logger.info("[3d] Missing value imputation complete")
    return df


In [ ]:
# Apply feature engineering to train and test
train_fe = engineer_features(train, is_train=True)
test_fe = engineer_features(test, is_train=False)


In [ ]:
# 3e. One-Hot Encoding
cat_to_ohe = cat_cols + ['bmi_category', 'heart_rate_zone']
logger.info(f"OHE categories: {cat_to_ohe}")

train_fe = pd.get_dummies(train_fe, columns=cat_to_ohe, drop_first=False, dummy_na=False)
test_fe = pd.get_dummies(test_fe, columns=cat_to_ohe, drop_first=False, dummy_na=False)

# Align columns between train and test (ensure same OHE columns)
train_fe, test_fe = train_fe.align(test_fe, join='inner', axis=1)
logger.info(f"After OHE - Train: {train_fe.shape}, Test: {test_fe.shape}")

# Drop id column (sequential index - no predictive power)
id_cols = [c for c in train_fe.columns if c == 'id']
if id_cols:
    train_fe = train_fe.drop(columns=id_cols)
    test_fe = test_fe.drop(columns=id_cols)
    logger.info("Dropped id column")

logger.info(f"Final features - Train: {train_fe.shape}, Test: {test_fe.shape}")
logger.info(f"Feature list:\n{list(train_fe.columns)}")

# Prepare model inputs
X = train_fe.values
y = train['health_condition'].values
X_test = test_fe.values
ids_test = test['id'].values

logger.info(f"\nX shape: {X.shape}")
logger.info(f"X_test shape: {X_test.shape}")
logger.info(f"Target classes: {np.unique(y)}")


## Step 4: CatBoost 5-Fold Cross-Validation

### Model Configuration
- **loss_function=MultiClass**: 3-class classification
- **auto_class_weights=Balanced**: Handle 85.9%/8.4%/5.8% imbalance
- **task_type=GPU**: GPU acceleration
- **early_stopping_rounds=50 + use_best_model=True**: Stop at convergence, revert to best
- Default for all other params: depth=6, lr=auto, iterations=1000

### Evaluation
- **Balanced Accuracy**: Primary metric (mean recall per class)
- **Confusion Matrix**: Per-class performance


In [ ]:
# ============================================================
# 4. CATBOOST 5-FOLD CROSS-VALIDATION
# ============================================================
logger.info("=" * 60)
logger.info("STEP 4: CATBOOST 5-FOLD CV")
logger.info("=" * 60)

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder

# Encode target
le = LabelEncoder()
y_enc = le.fit_transform(y)
logger.info(f"Label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

cv_scores = []
oof_preds = np.zeros(len(y_enc))
test_preds = np.zeros((len(X_test), len(le.classes_)))

# CatBoost parameters
cb_params = {
    'loss_function': 'MultiClass',
    'auto_class_weights': 'Balanced',
    'random_seed': SEED,
    'task_type': 'GPU',
    'thread_count': -1,
    'verbose': 50,
    'early_stopping_rounds': 50,
}
# All other params use defaults: depth=6, lr=auto, iterations=1000

logger.info(f"CatBoost params: {cb_params}")


In [ ]:
# 5-fold CV loop
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_enc)):
    logger.info(f"\n{'='*40}")
    logger.info(f"FOLD {fold + 1}/{n_splits}")
    logger.info(f"{'='*40}")
    
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y_enc[train_idx], y_enc[val_idx]
    
    logger.info(f"Train size: {X_tr.shape[0]}, Val size: {X_val.shape[0]}")
    logger.info(f"Train class dist: {np.bincount(y_tr)}")
    logger.info(f"Val class dist:   {np.bincount(y_val)}")
    
    model = CatBoostClassifier(**cb_params)
    
    start_time = datetime.now()
    model.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        use_best_model=True,
        logging_level='Verbose',
    )
    elapsed = (datetime.now() - start_time).total_seconds()
    best_iter = model.best_iteration_
    logger.info(f"Training time: {elapsed:.1f}s (best iteration: {best_iter})")
    
    # Validation predictions
    val_prob = model.predict_proba(X_val)
    val_pred = val_prob.argmax(axis=1)
    oof_preds[val_idx] = val_pred
    
    bal_acc = balanced_accuracy_score(y_val, val_pred)
    cv_scores.append(bal_acc)
    logger.info(f"Fold {fold + 1} Balanced Accuracy: {bal_acc:.6f}")
    
    # Test predictions (averaged across folds)
    test_prob = model.predict_proba(X_test)
    test_preds += test_prob / n_splits
    
    # Top 10 feature importances
    imp = pd.DataFrame({
        'feature': train_fe.columns,
        f'importance_fold{fold+1}': model.feature_importances_
    }).sort_values(f'importance_fold{fold+1}', ascending=False)
    logger.info(f"\nTop 10 features fold {fold+1}:\n{imp.head(10).to_string(index=False)}")


In [ ]:
# Aggregate CV results
logger.info(f"\n{'='*60}")
logger.info(f"CV RESULTS - {n_splits}-FOLD STRATIFIED")
logger.info(f"{'='*60}")
logger.info(f"Per-fold balanced accuracies: {[f'{s:.6f}' for s in cv_scores]}")
logger.info(f"Mean balanced accuracy: {np.mean(cv_scores):.6f} +/- {np.std(cv_scores):.6f}")

oof_bal_acc = balanced_accuracy_score(y_enc, oof_preds.astype(int))
logger.info(f"OOF Balanced Accuracy: {oof_bal_acc:.6f}")

from sklearn.metrics import confusion_matrix, classification_report
cm = confusion_matrix(y_enc, oof_preds.astype(int))
logger.info(f"\nOOF Confusion Matrix:\n{cm}")
logger.info(f"\nOOF Classification Report:\n{classification_report(y_enc, oof_preds.astype(int), target_names=le.classes_)}")


## Step 5: Generate Submission

Average predictions from 5 folds and save submission file.


In [ ]:
# ============================================================
# 5. SUBMISSION
# ============================================================
logger.info("=" * 60)
logger.info("STEP 5: GENERATING SUBMISSION")
logger.info("=" * 60)

test_labels = le.inverse_transform(test_preds.argmax(axis=1))
sub = pd.DataFrame({'id': ids_test, 'health_condition': test_labels})
sub.to_csv(os.path.join(OUTPUT_DIR, 'submission.csv'), index=False)

logger.info(f"Submission saved to submission.csv")
logger.info(f"Shape: {sub.shape}")
logger.info(f"Submission head:\n{sub.head(10)}")
logger.info(f"Submission distribution:\n{sub['health_condition'].value_counts()}")

logger.info("\n" + "=" * 60)
logger.info("DONE")
logger.info("=" * 60)
